# ReportBuilder Demo

This notebook demonstrates how to use `ModelMonitor` to compute drift metrics and then generate a human‑readable report with `ReportBuilder`.

## 1. Setup
Import required modules and create synthetic datasets.

In [1]:
import numpy as np
import pandas as pd
from momo_ml.monitor.model_monitor import ModelMonitor
from momo_ml.report.report_builder import ReportBuilder

# Set random seed for reproducibility
np.random.seed(42)

## 2. Create synthetic reference and current datasets

In [2]:
n_ref = 1000
n_cur = 1000

# Reference dataset
ref_df = pd.DataFrame({
    'feature_1': np.random.normal(0, 1, n_ref),
    'feature_2': np.random.uniform(0, 10, n_ref),
    'feature_3': np.random.choice(['A', 'B', 'C'], n_ref, p=[0.5, 0.3, 0.2]),
    'prediction': np.random.uniform(0, 1, n_ref),
    'label': np.random.choice([0, 1], n_ref, p=[0.7, 0.3])
})

# Current dataset with slight distribution shift
cur_df = pd.DataFrame({
    'feature_1': np.random.normal(0.3, 1.2, n_cur),          # shifted mean and std
    'feature_2': np.random.uniform(1, 12, n_cur),            # shifted range
    'feature_3': np.random.choice(['A', 'B', 'C'], n_cur, p=[0.4, 0.4, 0.2]),  # changed proportions
    'prediction': np.random.uniform(0.1, 0.9, n_cur),        # shifted predictions
    'label': np.random.choice([0, 1], n_cur, p=[0.6, 0.4])   # changed label distribution
})

print("Reference dataset shape:", ref_df.shape)
print("Current dataset shape:", cur_df.shape)
ref_df.head()

Reference dataset shape: (1000, 5)
Current dataset shape: (1000, 5)


,feature_1,feature_2,feature_3,prediction,label
0,0.496714,1.674826,A,0.648318,0
1,-0.138264,1.045678,A,0.074105,0
2,0.647689,6.364302,A,0.375469,0
3,1.523030,7.064757,A,0.803815,0
4,-0.234153,0.315861,C,0.433472,0


## 3. Run ModelMonitor
Compute performance drift, data drift, and prediction drift.

In [3]:
monitor = ModelMonitor(
    ref_df=ref_df,
    cur_df=cur_df,
    label_col='label',
    pred_col='prediction',
    data_drift_kwargs={'kl_buckets': 10},
    prediction_drift_kwargs={'include_psi': True, 'include_ks': True, 'include_kl': True, 'include_js': True}
)

results = monitor.run_all()

# Show top-level keys
print("Available result sections:", list(results.keys()))

Available result sections: ['performance_drift', 'data_drift', 'prediction_drift']


## 4. Build report with ReportBuilder

In [4]:
# Optional: custom thresholds for risk badges
custom_thresholds = {
    'psi': {'low': 0.1, 'medium': 0.25},
    'kl': {'low': 0.1, 'medium': 0.3},
    'js': {'low': 0.05, 'medium': 0.15}
}

builder = ReportBuilder(results, thresholds=custom_thresholds)

### 4.1 Generate Markdown report

In [5]:
markdown_report = builder.to_markdown(title="Monitoring Report - Demo")
print(markdown_report[:2000])  # Print first 2000 characters

# Save to file
builder.save_markdown("monitoring_report.md")
print("\nMarkdown report saved to 'monitoring_report.md'")

# Monitoring Report - Demo

### Performance drift (classification)
*Subtype: binary*
| Metric | Reference | Current | Delta |
|--------|-----------|---------|-------|
| accuracy | 0.4940 | 0.4960 | 0.0020 |
| auc | 0.4685 | 0.4967 | 0.0282 |
| f1 | 0.3595 | 0.4312 | 0.0717 |
| ks | 0.0624 | 0.0582 | -0.0042 |
| precision | 0.2874 | 0.3882 | 0.1008 |
| recall | 0.4797 | 0.4848 | 0.0050 |

### Data drift
#### Numeric features
| Feature | PSI | KS | KL | JS | WD |
|---------|-----|----|----|----|----|
| label | 🟢 0.0000 | 0.0980 | 🟢 0.0000 | 🟢 0.0000 | 0.0980 |
| feature_1 | 🟡 0.1138 | 0.1450 | 🟢 0.0555 | 🟢 0.0141 | 0.2891 |
| feature_2 | 🔴 1.6359 | 0.1810 | 🔴 2.4467 | 🟢 0.0383 | 1.3552 |
| prediction | 🔴 1.8794 | 0.1080 | 🔴 2.6015 | 🟡 0.0635 | 0.0582 |

#### Categorical features
| Feature | PSI | KL | JS | WD |
|---------|-----|----|----|----|
| feature_3 | 🟢 0.0465 | 🟢 0.0229 | 🟢 0.0058 | 0.0990 |

### Prediction drift (continuous)
#### Summary statistics
| Statistic | Reference | Curre

### 4.2 Generate JSON report

In [6]:
json_report = builder.to_json(indent=2)
print(json_report[:1000] + "...")  # Print first 1000 characters

# Save to file
builder.save_json("monitoring_report.json")
print("\nJSON report saved to 'monitoring_report.json'")

{
  "timestamp": "2026-03-25T16:46:47.503725",
  "results": {
    "performance_drift": {
      "task_type": "classification",
      "classification_subtype": "binary",
      "reference": {
        "auc": 0.4684524723587224,
        "ks": 0.06242321867321865,
        "accuracy": 0.494,
        "precision": 0.2874493927125506,
        "recall": 0.4797297297297297,
        "f1": 0.3594936708860759
      },
      "current": {
        "auc": 0.49665778760617185,
        "ks": 0.05820810507446683,
        "accuracy": 0.496,
        "precision": 0.3882113821138211,
        "recall": 0.4847715736040609,
        "f1": 0.43115124153498874
      },
      "delta": {
        "auc": 0.02820531524744946,
        "f1": 0.07165757064891282,
        "ks": -0.0042151135987518185,
        "precision": 0.10076198940127051,
        "recall": 0.005041843874331209,
        "accuracy": 0.0020000000000000018
      }
    },
    "data_drift": {
      "numeric_features": {
        "label": {
          "psi": 0.0,


## 5. Conclusion

You now have a complete workflow for monitoring model drift and generating reports. The Markdown report is easy to read, while the JSON file can be used for further integration.